### Data Ingestion


In [1]:
from langchain_core.documents import Document
from langchain_community.document_loaders import DirectoryLoader
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader

dir_loader = DirectoryLoader(
    "../data/pdf",
    glob="**/*.pdf", 
    loader_cls=PyMuPDFLoader,
    show_progress=False
)

pdf_documents=dir_loader.load()
pdf_documents

d:\Rag system - Copy\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[Document(metadata={'producer': 'pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'creator': 'LaTeX with acmart 2021/09/24 v1.80 Typesetting articles for the Association for Computing Machinery and hyperref 2023-04-22 v7.00x Hypertext links for LaTeX', 'creationdate': '2024-07-16T00:48:00+00:00', 'source': '..\\data\\pdf\\Ji et al. (2023).pdf', 'file_path': '..\\data\\pdf\\Ji et al. (2023).pdf', 'total_pages': 59, 'format': 'PDF 1.5', 'title': 'Survey of Hallucination in Natural Language Generation', 'author': '', 'subject': '-  Computing methodologies  ->  Natural language generation.Neural networks.', 'keywords': '', 'moddate': '2024-07-16T00:48:00+00:00', 'trapped': '', 'modDate': 'D:20240716004800Z', 'creationDate': 'D:20240716004800Z', 'page': 0}, page_content='Survey of Hallucination in Natural Language Generation\nZIWEI JI, NAYEON LEE, RITA FRIESKE, TIEZHENG YU, DAN SU, YAN XU, ETSUKO ISHII,\nYEJIN BANG, DELONG CHEN, WENLIANG DAI, HO SHU CHAN, AND

In [2]:
type(pdf_documents)

list

In [3]:
from langsmith import Client

client = Client()

dataset_name = "Alex_Management_Benchmark_v7"

if not client.has_dataset(dataset_name=dataset_name):
    dataset = client.create_dataset(
        dataset_name,
        description="Alex Advisor golden Q&A pairs — v7, new knowledge base (Atomic Habits, Rich Dad, Principles, Random Walk, Persuasion, Ready Fire Aim, Obstacle)."
    )
    print(f"Dataset '{dataset_name}' successfully created!")
else:
    print(f"Dataset '{dataset_name}' already exists.")

examples = [
    {
        "question": "A manager is frustrated that her team keeps missing their quarterly OKRs despite everyone agreeing on the targets. According to the book, what is the real problem and what should she focus on instead?",
        "ground_truth": "Clear argues the problem is the system, not the people or the goals. 'You do not rise to the level of your goals. You fall to the level of your systems.' Bad habits repeat themselves not because people don't want to change but because they have the wrong system for change. Focusing on the overall system rather than a single goal is one of the core themes of the book — the goal of a manager should be to design a system that makes the right behaviours automatic and reliable, rather than relying on willpower or motivation to hit a target.",
        "reference_ids": ["Atomic_Habits_Page_26"]
    },
    {
        "question": "A senior executive wants to build a habit of doing a five-minute team check-in every morning. What technique does the book recommend for reliably anchoring a new habit to a specific time and behaviour, and what is the formula for it?",
        "ground_truth": "Clear recommends implementation intentions combined with habit stacking. An implementation intention commits you to a specific time and location: 'I will [BEHAVIOUR] at [TIME] in [LOCATION].' Habit stacking then links a new behaviour to an existing one: 'After [CURRENT HABIT], I will [NEW HABIT].' For example: 'After I pour my morning coffee, I will send a one-sentence check-in message to my team.' The key is to tie the desired behaviour to something already done each day so the existing habit becomes the cue for the new one.",
        "reference_ids": ["Atomic_Habits_Page_61", "Atomic_Habits_Page_63"]
    },
    {
        "question": "What are the Four Laws of Behavior Change described in the book, and how should a team lead invert them to eliminate a counterproductive habit like defaulting to long email threads instead of short direct conversations?",
        "ground_truth": "The Four Laws for building a good habit are: (1) Make it obvious, (2) Make it attractive, (3) Make it easy, (4) Make it satisfying. To break a bad habit, invert each law. To eliminate reflexive long emails: make it invisible (remove email as the default communication tool for the team, disable desktop notifications); make it unattractive (frame direct conversation as the professional norm); make it difficult (add friction — require approval for emails over 100 words to the same team); make it unsatisfying (track and share email volume publicly so over-emailing has social cost). The Four Laws provide a simple set of rules for both creating good habits and eliminating bad ones.",
        "reference_ids": ["Atomic_Habits_Page_48", "Atomic_Habits_Page_50"]
    },
    {
        "question": "A manager is debating whether buying a company car for personal use counts as building wealth. How does the book define an asset versus a liability, and which category does a personal vehicle fall into?",
        "ground_truth": "Kiyosaki's Rich Dad states: 'Rule #1: You must know the difference between an asset and a liability, and buy assets.' His simple definition is: 'An asset puts money in my pocket. A liability takes money out of my pocket.' A personal vehicle is a liability — it depreciates, requires insurance, maintenance, and fuel, and generates no income. The book warns that most people — including the middle class — acquire liabilities they mistakenly think are assets: 'Rich people acquire assets. The poor and middle class acquire liabilities that they think are assets.'",
        "reference_ids": ["Rich_Dad_Page_60", "Rich_Dad_Page_61"]
    },
    {
        "question": "A high-earning director says her salary keeps increasing but she never seems to get ahead financially. What pattern does the book identify that explains this, and what does the financial statement of someone trapped in it look like?",
        "ground_truth": "Kiyosaki calls this the Rat Race. His poor dad's personal financial statement illustrates it: income and expenses are equal, meaning every raise is immediately consumed by higher spending, taxes, and debt, leaving nothing to invest. As expenses rise in lockstep with income — through bigger mortgages, car payments, and consumer debt — the liabilities column grows larger than the asset column. The person remains permanently dependent on their paycheck. The book shows that escaping the Rat Race requires using income to acquire assets, which then generate the cash flow to cover expenses, rather than spending every earned dollar on liabilities.",
        "reference_ids": ["Rich_Dad_Page_80", "Rich_Dad_Page_81"]
    },
    {
        "question": "What does the book say is the fundamental difference in how the poor, the middle class, and the rich manage the flow of money between income, expenses, assets, and liabilities?",
        "ground_truth": "The book presents three distinct cash-flow patterns. The poor spend all their income directly on expenses, leaving nothing to invest. The middle class earns income from a job, then immediately depletes it through expenses and the acquisition of liabilities — mortgages, car loans, credit-card debt — that they mistakenly think are assets. The rich acquire assets first; those assets generate income (passive and portfolio income), and that income then covers their expenses. The arrows in the cash-flow diagrams make this visible: the rich have income flowing from assets, not from labour alone. 'If you want to be rich, simply spend your life buying assets.'",
        "reference_ids": ["Rich_Dad_Page_62", "Rich_Dad_Page_63", "Rich_Dad_Page_64"]
    },
    {
        "question": "Dalio describes his most fundamental principle as the starting point for all effective decision-making. What is it, and why does he argue that ignoring it is the root cause of most poor outcomes?",
        "ground_truth": "Dalio's most fundamental principle is: 'Truth — more precisely, an accurate understanding of reality — is the essential foundation for producing good outcomes.' He argues that people who confuse what they wish were true with what is really true create distorted pictures of reality that make it impossible to make the best choices. By not facing harsh realities, they cannot find ways to properly deal with them, and because their decisions are not grounded in what is real, they cannot anticipate the consequences. He contrasts this with people who understand that knowing what is real is the first step toward optimally dealing with it, and who therefore make better decisions.",
        "reference_ids": ["Principles_Ray_Dalio_Page_19"]
    },
    {
        "question": "A manager keeps repeating the same resource-allocation mistake each quarter. Dalio has a specific formula for converting repeated failures into genuine progress — what is it and what does each component require in practice?",
        "ground_truth": "Dalio's formula is: Pain + Reflection = Progress. Nature uses pain as a messaging device to signal that limits have been reached. Most people react with fight-or-flight, which means they fail to learn. Those who develop 'a knee-jerk reaction to pain that is to reflect rather than to fight or flee' experience rapid learning. Reflection requires thinking deeply about what caused the pain, identifying the root cause (not the proximate cause), and then writing down the resulting principle so it is not forgotten. Dalio states this twice in the book: 'If there is only one piece of advice I can get you to remember it is this one' — when you experience pain, reflect.",
        "reference_ids": ["Principles_Ray_Dalio_Page_27", "Principles_Ray_Dalio_Page_47"]
    },
    {
        "question": "What are the two internal barriers Dalio says block most people from seeing reality clearly and making good decisions, and where do these barriers originate neurologically?",
        "ground_truth": "Dalio identifies the ego barrier and the blind-spot barrier. The ego barrier is rooted in defensive, emotional reactions that take place in the amygdala. When someone points out a weakness or mistake, the brain produces fight-or-flight reactions that make people feel attacked, causing them to avoid reflection on their own weaknesses — which is, Dalio argues, 'the biggest single problem of mankind.' The blind-spot barrier refers to the areas everyone has where they simply cannot see clearly because of the design of their own mind. Together, these barriers mean most people do not objectively understand themselves and others, which impedes their ability to get what they want out of life.",
        "reference_ids": ["Principles_Ray_Dalio_Page_24"]
    },
    {
        "question": "A board is considering hiring an active fund manager for the company pension. The book makes a central argument about whether professional managers can consistently outperform the market — what is it and what evidence does the author use?",
        "ground_truth": "Malkiel argues that buying a broad index fund — 'buying the haystack itself' — is a strategy far more likely to be optimal than trying to pick a needle (an outperforming fund). He shows that no sizable differences in investment performance exist among professionally managed portfolios as a group, including mutual funds, pension funds, insurance companies, and bank trust accounts, compared to the broad market. Exceptions are 'very rare.' No scientific evidence has been assembled showing that professionally managed portfolios as a group have performed better than a broad-based index. His recommendation is direct: invest in a low-cost, passively managed index fund.",
        "reference_ids": ["Random_Walk_Page_140", "Random_Walk_Page_141"]
    },
    {
        "question": "An analyst on your team swears by studying price charts to predict when to buy and sell equities. How does the book characterise technical analysis and what conclusion does it reach about its effectiveness?",
        "ground_truth": "Malkiel describes technical analysis as the making and interpreting of stock charts — practitioners are called chartists. They study past price movements and trading volume to predict future price direction. The first principle of technical analysis is that all information, including fundamentals, is already reflected in past market prices. The second is that prices tend to move in trends. Malkiel's conclusion is that technical analysis does not consistently produce returns that outperform the market, because in an efficient market, past prices contain no exploitable information about future prices. He argues the approach is theoretically undermined by the random walk hypothesis: stock price changes from period to period are largely independent.",
        "reference_ids": ["Random_Walk_Page_83", "Random_Walk_Page_84"]
    },
    {
        "question": "The book examines fundamental analysis as the alternative to charting. What conclusion does Malkiel reach about it, and what does he say even Benjamin Graham eventually concluded about the technique he pioneered?",
        "ground_truth": "Malkiel acknowledges that fundamental analysis — finding a stock's intrinsic value based on assets, expected earnings growth, dividends, interest rates, and risk — is more theoretically grounded than technical analysis. However, he concludes it is no better than technical analysis at enabling investors to capture above-average returns. He cites the Efficient Market Hypothesis: all public information is already reflected in stock prices, so no analyst can consistently identify mispriced securities before the market corrects them. He quotes Benjamin Graham — the father of fundamental security analysis — who said shortly before his death in 1976: 'I am no longer an advocate of elaborate techniques of security analysis in order to find superior value opportunities.'",
        "reference_ids": ["Random_Walk_Page_140", "Random_Walk_Page_141", "Random_Walk_Page_142"]
    },
    {
        "question": "A sales team wants to understand why sending a small free report to prospects before calling them dramatically increases close rates. Which of Cialdini's principles explains this, and what is the psychological mechanism behind it?",
        "ground_truth": "This is the principle of reciprocation. Cialdini explains that there is no human society that does not subscribe to the rule of reciprocity — it is a deeply ingrained obligation to repay what another person has provided. The rule is so powerful it can produce exchanges that are profoundly unequal: people will give back more than they received in order to discharge the feeling of indebtedness. The book illustrates this with the example of Ethiopia sending $5,000 in disaster relief to Mexico in 1985 — despite Ethiopia's own desperate poverty — because Mexico had sent aid to Ethiopia in 1935. The key insight for practitioners is that the gift or favour does not need to be solicited or even wanted to activate the rule; uninvited debts still produce the obligation.",
        "reference_ids": ["Psychology_Persuasion_Page_24", "Psychology_Persuasion_Page_25", "Psychology_Persuasion_Page_26"]
    },
    {
        "question": "During a project retrospective, a team member who originally championed a now-failing approach continues to defend it stubbornly despite clear evidence it isn't working. Which principle does the book say is at work, and how does it operate?",
        "ground_truth": "This is the principle of commitment and consistency. Once a person takes a stand or makes a commitment — especially publicly — they feel internal and external pressure to behave consistently with that prior position, because people need to be seen (and to see themselves) as consistent and rational. The book documents how the Chinese systematically exploited this in Korean War POW camps: they began by getting American prisoners to make small, seemingly innocuous written concessions, which then became the foundation for extracting larger collaborations. The commitment constrained future behaviour not through external force but through the prisoner's own need to be consistent with what they had already done and said.",
        "reference_ids": ["Psychology_Persuasion_Page_53", "Psychology_Persuasion_Page_54"]
    },
    {
        "question": "What does the book mean by social proof, and under what two specific conditions does Cialdini say it has its strongest influence on behaviour?",
        "ground_truth": "Social proof is the principle that 'one means we use to determine what is correct is to find out what other people think is correct.' The book states: 'We view a behavior as more correct in a given situation to the degree that we see others performing it.' Cialdini illustrates it with canned television laughter — audiences laugh more and rate material as funnier when a laugh track plays, even though they know the laughter is fabricated, because the sound of others laughing triggers the social proof heuristic automatically. The two conditions under which social proof is strongest are: uncertainty (when people genuinely do not know what the right action is) and similarity (when the reference group is perceived as similar to oneself).",
        "reference_ids": ["Psychology_Persuasion_Page_88", "Psychology_Persuasion_Page_89"]
    },
    {
        "question": "A founder has a business generating $6 million in revenue but is barely breaking even and has been at this revenue level for two years. According to the book's four-stage framework, what stage is this business in, what is its primary problem, and what must the founder do next?",
        "ground_truth": "The business is in Stage Two: Childhood, which spans $1 million to $10 million in revenue. The book defines this stage as: 'Main Problem: You are only breaking even or may even be losing money. Main Challenge: Creating many additional, profitable products quickly. Main Opportunity: Increasing cash flow and becoming profitable.' The founder must develop the skill of generating a constant stream of new and potentially tipping-point ideas for products. The first product alone cannot support the overhead at this stage — a portfolio of offers is required to achieve profitability and generate the cash flow needed to sustain and grow the business.",
        "reference_ids": ["Ready_Fire_Aim_Page_36", "Ready_Fire_Aim_Page_64"]
    },
    {
        "question": "A first-time entrepreneur is spending his first weeks setting up office space, designing a logo, and ordering inventory before making a single sale. What does the book say is Rule Number One of Entrepreneurship and what should this founder actually be doing?",
        "ground_truth": "Masterson calls it the Supremacy of Selling: 'Without sales, it is very hard to sustain an ongoing business. Consider this to be Rule Number One of Entrepreneurship.' Before the first sale, a business is 'nothing more than a set of unproven ideas that you are spending money on.' The book uses Jim Koch of The Boston Beer Company as an example — Koch spent his first week shopping for a computer before his uncle demanded: 'You know, Jim, I've seen a lot more businesses go broke because they didn't have enough sales than I've seen go under from lack of computers. Why don't you work on first things first?' Masterson's prescription is to eliminate all secondary activities and focus entirely on making the first profitable sale.",
        "reference_ids": ["Ready_Fire_Aim_Page_64", "Ready_Fire_Aim_Page_66"]
    },
    {
        "question": "What is the Optimum Selling Strategy (OSS) as Masterson defines it, what are the four questions that determine it, and what initial guidance does the book give a new entrepreneur trying to identify theirs?",
        "ground_truth": "The book describes the OSS as the single best way for a business at any given stage to acquire new customers. The four questions the entrepreneur must answer are: (1) Where are your customers? (2) What product(s) to sell them? (3) How much to charge? (4) How to convince them to buy? The book's blunt initial guidance for answering all four is: 'Do what everyone else is doing' — study competitors, identify where they advertise consistently, what they charge, and what offers they use. The best advertising locations for competitors are probably the best locations for you. Once cash flow is established in Stage One, the entrepreneur can then begin to 'test away from' the initial OSS to find improvements.",
        "reference_ids": ["Ready_Fire_Aim_Page_94", "Ready_Fire_Aim_Page_97"]
    },
    {
        "question": "A business unit leader has just lost a major client and is paralysed. What are the three disciplines the book prescribes for effectively confronting this kind of obstacle, and what does each one require?",
        "ground_truth": "The book structures its entire framework around three disciplines drawn from Stoic philosophy. Part I: Perception — seeing the obstacle clearly and objectively, without emotional distortion, and identifying within it any opportunity or advantage. Part II: Action — taking persistent, creative, directed action on what is within your control, iterating and adjusting rather than freezing. Part III: Will — building the inner strength to endure and accept what cannot be changed. The book describes these as the discipline of the mind, the body, and the heart. Together they form a complete system: 'First, see clearly. Next, act correctly. Finally, endure and accept the world as it is.'",
        "reference_ids": ["Obstacle_Page_9", "Obstacle_Page_10", "Obstacle_Page_11"]
    },
    {
        "question": "What is the Marcus Aurelius quote that anchors the book's central argument, and what practical principle does the author derive from it for someone facing a constraint they cannot remove?",
        "ground_truth": "The central quote from Marcus Aurelius is: 'The impediment to action advances action. What stands in the way becomes the way.' Holiday derives from this the principle of the 'reverse clause' — the idea that every obstacle contains within it the seed of an advantage, and that setbacks and problems should never be treated as permanent. The constraint that cannot be removed forces creativity, builds capability, and redirects energy in ways that a clear path never would. What impedes us can empower us. Holiday presents this not as positive thinking but as a literal strategy: the obstacle becomes the material from which progress is made.",
        "reference_ids": ["Obstacle_Page_9"]
    },
    {
        "question": "What does the book mean by 'amor fati,' and how does it differ from simple resignation or stoic endurance? What example does Holiday use to illustrate it?",
        "ground_truth": "Amor fati — 'love of fate' — is defined by Nietzsche (quoted in the book) as wanting 'nothing to be different, not forward, not backward, not in all eternity. Not merely bear what is necessary, still less conceal it... but love it.' The book contrasts it with mere acceptance or endurance: those approaches tolerate adversity while remaining inwardly resistant. Amor fati means actively choosing to treat every event, including disasters, as something useful and necessary. Holiday illustrates it with Thomas Edison: when fire destroyed his research campus and much of his life's work, Edison calmly told his son, 'Go get your mother and all her friends. They'll never see a fire like this again.' He called the loss 'a lot of rubbish' and within three weeks had the factory running again. Within the year, he generated nearly $10 million in revenue.",
        "reference_ids": ["Obstacle_Page_130", "Obstacle_Page_131", "Obstacle_Page_132", "Obstacle_Page_133"]
    }
]

client.create_examples(
    inputs=[{"question": ex["question"]} for ex in examples],
    outputs=[{
        "ground_truth": ex["ground_truth"],
        "reference_ids": ex.get("reference_ids", [])
    } for ex in examples],
    dataset_id=client.read_dataset(dataset_name=dataset_name).id
)
print("Golden examples successfully uploaded!")


Dataset 'Alex_Management_Benchmark_v6' successfully created!
Golden examples successfully uploaded!
